In [1]:
import os, sys
sys.path.append('../')

# 04 模型模块 (core.models)

提供风控建模相关的模型和工具。

**数据说明**: 基于 `hscredit_yyp.xlsx`，目标变量为 `MOB1 > 3`

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from hscredit import init_setting
from hscredit.core.models import (
    XGBoostRiskModel, LightGBMRiskModel,
    LogisticRegression, ScoreCard
)
from hscredit.core.metrics import ks, auc

init_setting()

# 加载数据
df = pd.read_excel('hscredit_yyp.xlsx')

# 构造目标变量
df['target'] = (df['MOB1'] > 3).astype(int)

# 选择数值特征
numeric_features = ['中智小牛分C3', '珊瑚92', '极光欺诈分6v1', '青云24', '占信V3',
                   '轻花老客海纳子分V1', '天创小额网贷分', '衡枢鉴真分老客版']

# 处理缺失值
df_model = df[numeric_features + ['target']].copy()
df_model = df_model.dropna()

X = df_model[numeric_features]
y = df_model['target']

print(f"样本数: {len(df_model):,}")
print(f"特征数: {len(numeric_features)}")
print(f"坏样本率: {y.mean():.2%}")

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"训练集: {len(X_train):,}, 测试集: {len(X_test):,}")

训练集: 184, 测试集: 80


## 1. XGBoost模型

In [ ]:
# 训练XGBoost模型（XGBoost可选依赖）
try:
    xgb_model = XGBoostRiskModel(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        min_child_weight=50,
        random_state=42
    )
    xgb_model.fit(X_train, y_train)
    y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
    print(f"XGBoost KS: {ks(y_test, y_prob_xgb):.4f}")
    print(f"XGBoost AUC: {auc(y_test, y_prob_xgb):.4f}")
except ImportError as e:
    print(f"XGBoost未安装: {e}")
    y_prob_xgb = None

## 2. LightGBM模型

In [5]:
# 训练LightGBM模型
lgb_model = LightGBMRiskModel(
    n_estimators=100,
    num_leaves=31,
    learning_rate=0.1,
    random_state=42
)

lgb_model.fit(X_train, y_train)

# 预测
y_prob_lgb = lgb_model.predict_proba(X_test)[:, 1]

print(f"LightGBM KS: {ks(y_test, y_prob_lgb):.4f}")
print(f"LightGBM AUC: {auc(y_test, y_prob_lgb):.4f}")

NameError: name 'ks' is not defined

## 3. 逻辑回归模型

In [6]:
# 训练逻辑回归
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train, y_train)

# 预测
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

print(f"LR KS: {ks(y_test, y_prob_lr):.4f}")
print(f"LR AUC: {auc(y_test, y_prob_lr):.4f}")

# 查看系数
coef_df = pd.DataFrame({
    '特征': X.columns,
    '系数': lr_model.coef_[0]
}).sort_values('系数', key=abs, ascending=False)
display(coef_df)

NameError: name 'ks' is not defined

## 4. 评分卡模型

In [7]:
from hscredit.core.binning import OptimalBinning

# 对特征进行分箱
binner = OptimalBinning(method='best_iv', max_n_bins=5)
binner.fit(X_train, y_train)

X_train_binned = binner.transform(X_train)
X_test_binned = binner.transform(X_test)

# 创建评分卡
scorecard = ScoreCard(base_score=600, pdo=20)
scorecard.fit(X_train_binned, y_train)

# 预测分数
scores = scorecard.predict_score(X_test_binned)

print(f"评分范围: {scores.min():.0f} - {scores.max():.0f}")
print(f"平均分: {scores.mean():.0f}")

评分范围: 499 - 640
平均分: 569


## 5. 模型对比

In [ ]:
from hscredit.core.metrics import lift_at

# 构建模型对比表
model_rows = []

# LightGBM (始终可用)
model_rows.append({
    '模型': 'LightGBM',
    'KS': ks(y_test, y_prob_lgb),
    'AUC': auc(y_test, y_prob_lgb)
})

# XGBoost (可选)
if y_prob_xgb is not None:
    model_rows.append({
        '模型': 'XGBoost',
        'KS': ks(y_test, y_prob_xgb),
        'AUC': auc(y_test, y_prob_xgb)
    })

# LogisticRegression
model_rows.append({
    '模型': 'LogisticRegression',
    'KS': ks(y_test, y_prob_lr),
    'AUC': auc(y_test, y_prob_lr)
})

models_comparison = pd.DataFrame(model_rows)
display(models_comparison.round(4))